# PSD Diagnostics — Colab Runner

Runs `analysis/run_psd_diagnostics.py` from the project mounted on Google Drive.
Two modes are provided:
- **Synthetic** — no input data required; generates a deterministic sphere-packed volume for pipeline validation.
- **Real** — supply a 3-D binary pore volume (`.npy` or `.tif`/`.tiff`, `True = pore`).

All outputs are written to `<OUTPUT_ROOT>/psd_diag_<timestamp>_<run-name>/`.

## Step 0 — GPU Check

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    raise RuntimeError(
        'No GPU available. Go to Runtime > Change runtime type and select GPU.'
    )
print(result.stdout)

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted at /content/drive')

## Step 2 — Install Dependencies

`scipy` is required for CPU EDT. `tifffile` is required for `.tif` input.
`cupy-cuda12x` enables GPU EDT (skip if your Colab runtime uses a different CUDA version).

In [ ]:
!pip install -q scipy tifffile cupy-cuda12x

## Step 3 — Configure Paths

Edit `DRIVE_PROJECT_ROOT` to match the location of the project on your Drive.

In [ ]:
import os

# --- EDIT THIS ---
DRIVE_PROJECT_ROOT = '/content/drive/MyDrive/nnUNet4SoilXrayCT'
# -----------------

ANALYSIS_DIR = os.path.join(DRIVE_PROJECT_ROOT, 'analysis')
OUTPUT_ROOT  = os.path.join(DRIVE_PROJECT_ROOT, 'analysis', 'outputs')

os.makedirs(OUTPUT_ROOT, exist_ok=True)

print(f'ANALYSIS_DIR : {ANALYSIS_DIR}')
print(f'OUTPUT_ROOT  : {OUTPUT_ROOT}')
print('analysis/ contents:', os.listdir(ANALYSIS_DIR))

## Step 4 — Synthetic Mode (sanity check, no input data needed)

Generates a deterministic sphere-packed binary volume and runs the full PSD pipeline.
Output is written to `<OUTPUT_ROOT>/psd_diag_<timestamp>_synthetic_test/`.

In [ ]:
!cd "{ANALYSIS_DIR}" && python run_psd_diagnostics.py synthetic \
    --output-root "{OUTPUT_ROOT}" \
    --run-name synthetic_test \
    --shape 80 80 80 \
    --voxel-spacing 1.0 1.0 1.0 \
    --sphere-count 40 \
    --seed 42 \
    --chunk-size 64 64 64 \
    --halo-width 32
# Output folder: <OUTPUT_ROOT>/psd_diag_<timestamp>_synthetic_test/

## Step 5 — Real Mode (binary pore volume)

Supply the path to a 3-D binary pore volume (`.npy` or `.tif`/`.tiff`; `True = pore`) 
and set the physical voxel spacing in micrometres `(dz, dy, dx)`.

Edit `INPUT_VOLUME` and `VOXEL_SPACING_UM` before running.

In [ ]:
# --- EDIT THESE ---
INPUT_VOLUME     = '/content/drive/MyDrive/nnUNet4SoilXrayCT/data/pore_volume.npy'
VOXEL_SPACING_UM = '2.0 2.0 2.0'   # dz dy dx  (micrometres)
RUN_NAME         = 'scan_001'
# ------------------

print(f'Input volume : {INPUT_VOLUME}')
print(f'Voxel spacing: {VOXEL_SPACING_UM} um')

In [ ]:
!cd "{ANALYSIS_DIR}" && python run_psd_diagnostics.py real \
    --input "{INPUT_VOLUME}" \
    --voxel-spacing {VOXEL_SPACING_UM} \
    --output-root "{OUTPUT_ROOT}" \
    --run-name "{RUN_NAME}"
# Output folder: <OUTPUT_ROOT>/psd_diag_<timestamp>_<RUN_NAME>/
# Artifacts: config.json, result_psd.json, diagnostics.json, summary.json, psd_table.csv

## Step 6 — Real Mode with Chunking (large volumes)

Use `--use-chunking` for volumes that do not fit in GPU memory.
Adjust `--chunk-size` and `--halo-width` as needed.

In [ ]:
!cd "{ANALYSIS_DIR}" && python run_psd_diagnostics.py real \
    --input "{INPUT_VOLUME}" \
    --voxel-spacing {VOXEL_SPACING_UM} \
    --output-root "{OUTPUT_ROOT}" \
    --run-name "{RUN_NAME}_chunked" \
    --use-chunking \
    --chunk-size 128 128 128 \
    --halo-width 50
# Output folder: <OUTPUT_ROOT>/psd_diag_<timestamp>_<RUN_NAME>_chunked/

## Step 7 — Inspect Outputs

List the latest run folder and print its `summary.json`.

In [ ]:
import json, pathlib

run_folders = sorted(pathlib.Path(OUTPUT_ROOT).glob('psd_diag_*'))
if not run_folders:
    print('No run folders found in', OUTPUT_ROOT)
else:
    latest = run_folders[-1]
    print(f'Latest run: {latest.name}')
    print('Files:', [f.name for f in sorted(latest.iterdir())])
    summary_path = latest / 'summary.json'
    if summary_path.exists():
        with open(summary_path) as fh:
            print(json.dumps(json.load(fh), indent=2))